In [1]:
# type: ignore

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


load_dotenv()

model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),
)

In [ ]:
from langchain_core.tools import tool


@tool
def write_file(filename: str, content: str) -> str:
    """
    Write content to a file.

    Args:
        filename: The path of the file to write.
        content: The content to write into the file.
    """
    print(f"[MOCK] Writing to file {filename} with content:\n{content}")
    return f"Successfully wrote to file {filename} (mock)"


@tool
def execute_sql(query: str) -> str:
    """
    Execute an SQL query (potentially dangerous operation).

    Args:
        query: The SQL query to execute.
    """
    print(f"[MOCK] Executing SQL query: {query}")
    if "SELECT" in query.upper():
        return "Query result: [(1, 'mock_data'), (2, 'another_row')] (mock)"
    elif "DELETE" in query.upper():
        return "Deleted 5 rows (mock, no real operation performed)"
    else:
        return "SQL executed successfully (mock)"


@tool
def read_data(table_name: str) -> str:
    """
    Read data from a table (safe operation).

    Args:
        table_name: The name of the table to read from.
    """
    print(f"[MOCK] Reading data from table {table_name}")
    return f"Data from table {table_name}:\n[{'id': 1, 'name': 'Alice'}, {'id': 2, 'name': 'Bob'}] (mock)"

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


agent = create_agent(
    model=model,
    tools=[write_file, execute_sql, read_data],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file": True,  # All decisions (approve, edit, reject) allowed
                "execute_sql": {
                    "allowed_decisions": ["approve", "reject"]
                },  # No editing allowed
                # Safe operation, no approval needed
                "read_data": False,
            },
            # Prefix for interrupt messages - combined with tool name and args to form the full message
            # e.g., "Tool execution pending approval: execute_sql with query='DELETE FROM...'"
            # Individual tools can override this by specifying a "description" in their interrupt config
            description_prefix="Tool execution pending approval",
        ),
    ],
    # Human-in-the-loop requires checkpointing to handle interrupts.
    # In production, use a persistent checkpointer like AsyncPostgresSaver.
    checkpointer=InMemorySaver(),
)

In [ ]:
from langchain_core.runnables import RunnableConfig


# Human-in-the-loop leverages LangGraph's persistence layer.
# We must provide a thread ID to associate the execution with a conversation thread,
# so the conversation can be paused and resumed (as is needed for human review).
config = RunnableConfig({"configurable": {"thread_id": "1"}})
# Run the graph until the interrupt is hit.
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "丛数据库中移除旧纪录",
            }
        ]
    },
    config=config,
)

result

{'messages': [HumanMessage(content='丛数据库中移除旧纪录', additional_kwargs={}, response_metadata={}, id='8b38faee-9159-445d-9896-5377ff6f3804'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 450, 'total_tokens': 479, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3-max-2025-09-23', 'system_fingerprint': None, 'id': 'chatcmpl-73a709a3-c42c-4533-aab7-82e1aebe130a', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--6d0424a8-6488-44a4-93ae-5dc9887fe6fe-0', tool_calls=[{'name': 'execute_sql', 'args': {'query': "DELETE FROM old_records WHERE condition = 'old'"}, 'id': 'call_d7908bf54deb4e918968051c', 'type': 'tool_call'}], usage_metadata={'input_tokens': 450, 'output_tokens': 29, 'total_tokens': 479, 'input_token_details': {}, 'output_token_details': {}})],
 '__interrupt__': [Interrupt(value={'action_requests': [{'name': 'execute_s

In [19]:
from rich import print as rprint


rprint(result["__interrupt__"])

[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'execute_sql',
                    'args': {'query': "DELETE FROM old_records WHERE condition = 'old'"},
                    'description': 'Tool execution pending approval\n\nTool: execute_sql\nArgs: {\'query\': "DELETE
FROM old_records WHERE condition = \'old\'"}'
                }
            ],
            'review_configs': [{'action_name': 'execute_sql', 'allowed_decisions': ['approve', 'reject']}]
        },
        id='9c0c7a2ac8900f6867fb0ddac50a3fb5'
    )
]

In [ ]:
from langgraph.types import Command


# Resume with approval decision
agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ),
    config=config,  # Same thread ID to resume the paused conversation
)

[MOCK] Executing SQL query: DELETE FROM old_records WHERE condition = 'old'


{'messages': [HumanMessage(content='丛数据库中移除旧纪录', additional_kwargs={}, response_metadata={}, id='8b38faee-9159-445d-9896-5377ff6f3804'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 450, 'total_tokens': 479, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3-max-2025-09-23', 'system_fingerprint': None, 'id': 'chatcmpl-73a709a3-c42c-4533-aab7-82e1aebe130a', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--6d0424a8-6488-44a4-93ae-5dc9887fe6fe-0', tool_calls=[{'name': 'execute_sql', 'args': {'query': "DELETE FROM old_records WHERE condition = 'old'"}, 'id': 'call_d7908bf54deb4e918968051c', 'type': 'tool_call'}], usage_metadata={'input_tokens': 450, 'output_tokens': 29, 'total_tokens': 479, 'input_token_details': {}, 'output_token_details': {}}),
  ToolMessage(content='Deleted 5 rows (mock, no real operation performed)', 

In [21]:
config = RunnableConfig({"configurable": {"thread_id": "2"}})
agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "丛数据库中移除旧纪录",
            }
        ]
    },
    config=config,
)

agent.invoke(Command(resume={"decisions": [{"type": "reject"}]}), config=config)

{'messages': [HumanMessage(content='丛数据库中移除旧纪录', additional_kwargs={}, response_metadata={}, id='09478b39-472d-43f1-8b45-d3347cc1185b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 450, 'total_tokens': 488, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3-max-2025-09-23', 'system_fingerprint': None, 'id': 'chatcmpl-a089ddfe-7ecc-4775-8c10-2fd3d117fef9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--20d4720e-a493-428d-a492-75045ccc3d25-0', tool_calls=[{'name': 'execute_sql', 'args': {'query': "DELETE FROM old_records WHERE date < '2020-01-01'"}, 'id': 'call_06fd1fa9c2954330beda0d4b', 'type': 'tool_call'}], usage_metadata={'input_tokens': 450, 'output_tokens': 38, 'total_tokens': 488, 'input_token_details': {}, 'output_token_details': {}}),
  ToolMessage(content='User rejected the tool call for `execute_sql` with 

In [23]:
config = RunnableConfig({"configurable": {"thread_id": "4"}})
agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "将'Hello'写入 hello.txt",
            }
        ]
    },
    config=config,
)

agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "write_file",
                        # Arguments to pass to the tool.
                        "args": {"filename": "hello.md", "content": "你好"},
                    },
                }
            ]
        }
    ),
    config=config,  # Same thread ID to resume the paused conversation
)

[MOCK] Writing to file hello.md with content:
你好


{'messages': [HumanMessage(content="将'Hello'写入 hello.txt", additional_kwargs={}, response_metadata={}, id='c046aeb5-519e-47f7-a809-facd89452f64'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 451, 'total_tokens': 484, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3-max-2025-09-23', 'system_fingerprint': None, 'id': 'chatcmpl-719e1069-309b-4719-9cd6-12b5e354d6ee', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--d641d223-e3b4-4518-927a-df169d076fea-0', tool_calls=[{'type': 'tool_call', 'name': 'write_file', 'args': {'filename': 'hello.md', 'content': '你好'}, 'id': 'call_400e3f88a3364b3c9b58ddd7'}], usage_metadata={'input_tokens': 451, 'output_tokens': 33, 'total_tokens': 484, 'input_token_details': {}, 'output_token_details': {}}),
  ToolMessage(content='Successfully wrote to file hello.md (mock)', name='write_file',

In [ ]:
# type: ignore

config = RunnableConfig({"configurable": {"thread_id": "9"}})

# Stream agent progress and LLM tokens until interrupt
for mode, chunk in agent.stream(
    {
        "messages": [
            {"role": "user", "content": "从数据库中删除旧记录并把结果写到log.info中"}
        ]
    },
    config=config,
    stream_mode=["updates", "messages"],
):
    if mode == "messages":
        # LLM token
        token, metadata = chunk
        if token.content:
            print(token.content, end="", flush=True)
    elif mode == "updates":
        # Check for interrupt
        if "__interrupt__" in chunk:
            for interrupt in chunk["__interrupt__"]:
                for action in interrupt.value["action_requests"]:
                    print(f"{action['description']}")

Tool execution pending approval

Tool: execute_sql
Args: {'query': "DELETE FROM records WHERE date < '2020-01-01'"}
Tool execution pending approval

Tool: write_file
Args: {'filename': 'log.info', 'content': 'Old records deleted successfully.'}


In [67]:
# type: ignore

# Resume with streaming after human decision
for mode, chunk in agent.stream(
    Command(
        resume={
            "decisions": [
                {"type": "approve"},
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "write_file",
                        # Arguments to pass to the tool.
                        "args": {"filename": "log.txt", "content": "删除完毕"},
                    },
                },
            ]
        }
    ),
    config=config,
    stream_mode=["updates", "messages"],
):
    if mode == "messages":
        token, metadata = chunk
        if token.content:
            print(token.content, end="", flush=True)

[MOCK] Executing SQL query: DELETE FROM records WHERE date < '2020-01-01'
[MOCK] Writing to file log.txt with content:
删除完毕
Deleted 5 rows (mock, no real operation performed)Successfully wrote to file log.txt (mock)

已成功从数据库中删除旧记录，并将操作结果写入 `log.txt` 文件中。

In [ ]:
config = RunnableConfig({"configurable": {"thread_id": "1112"}})

# Stream agent progress and LLM tokens until interrupt
for mode, event in agent.stream(
    {
        "messages": [
            {"role": "user", "content": "从数据库中删除旧记录并把结果写到log.info中"}
        ]
    },
    config=config,
    stream_mode=["updates", "messages"],
):
    if mode == "updates":
        print(event)
        for node, updates in event.items():
            if node == "__interrupt__":
                for interrupt in updates:
                    print(interrupt.value)

{'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'qwen3-max-2025-09-23', 'model_provider': 'openai'}, id='lc_run--6c921500-ffa1-44e8-beca-3c4d8906966f', tool_calls=[{'name': 'execute_sql', 'args': {'query': "DELETE FROM old_records WHERE created_at < '2020-01-01'"}, 'id': 'call_d0c7d2f90bd647d7a551d183', 'type': 'tool_call'}, {'name': 'write_file', 'args': {'filename': 'log.info', 'content': 'Old records deleted successfully.'}, 'id': 'call_c144da3517ba49de830c7cc2', 'type': 'tool_call'}])]}}
{'__interrupt__': (Interrupt(value={'action_requests': [{'name': 'execute_sql', 'args': {'query': "DELETE FROM old_records WHERE created_at < '2020-01-01'"}, 'description': 'Tool execution pending approval\n\nTool: execute_sql\nArgs: {\'query\': "DELETE FROM old_records WHERE created_at < \'2020-01-01\'"}'}, {'name': 'write_file', 'args': {'filename': 'log.info', 'content': 'Old records deleted successfully.'}, 'des

In [ ]:
config = RunnableConfig({"configurable": {"thread_id": "115"}})

# Stream agent progress and LLM tokens until interrupt
response_events = await agent.ainvoke(
    {
        "messages": [
            {"role": "user", "content": "从数据库中删除旧记录并把结果写到log.info中"}
        ]
    },
    config=config,
    stream_mode=["updates", "messages"],
)

response_type, response = response_events[-1]
if response_type == "updates" and "__interrupt__" in response:
    rprint(response["__interrupt__"])
    format_response = ""

(
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'execute_sql',
                    'args': {'query': "DELETE FROM old_records WHERE created_at < '2020-01-01'"},
                    'description': 'Tool execution pending approval\n\nTool: execute_sql\nArgs: {\'query\': "DELETE
FROM old_records WHERE created_at < \'2020-01-01\'"}'
                },
                {
                    'name': 'write_file',
                    'args': {'filename': 'log.info', 'content': 'Old records deleted successfully.'},
                    'description': "Tool execution pending approval\n\nTool: write_file\nArgs: {'filename': 
'log.info', 'content': 'Old records deleted successfully.'}"
                }
            ],
            'review_configs': [
                {'action_name': 'execute_sql', 'allowed_decisions': ['approve', 'reject']},
                {'action_name': 'write_file', 'allowed_decisions': ['approve', 'edit', 'reject']}
            ]
        },
        id='7e61ca57c0c37e62761bc97635de66a4'
    ),
)